In [2]:
# ============================================
# Radar-Based Pose Estimation (Project Version)
# ============================================

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# root = sessionID+'/alignment/'
root = './dataset/'
index = root + '00200'

# for plot
import matplotlib.pyplot as plt
import matplotlib.patches as patches
# joints connections for 2d keypoints
connections = np.array([[13, 15], [11, 13], [14, 16], [12, 14], [11, 12],
                [5, 11], [6, 12], [5, 6], [5, 7], [6, 8], [7, 9],
                [8, 10], [1, 2], [0, 1], [0, 2], [1, 3], [2, 4],
                [3, 5], [4, 6]])

# Meta info
data = np.load(index + '_meta.npz')
global_id = data['global_frame_id']
# Horizontal/Vertical heatmaps
data = np.load(index + '_radar.npz')
hori = data['hm_hori']  # (256, 128)
vert = data['hm_vert']  # (256, 128)
# 2D Bounding Boxes
data = np.load(index + '_bbox.npz')
bbox_i= data['bbox_i']  # (n, 5)
bbox_hori = data['bbox_hori']  # (n, 4)
bbox_vert = data['bbox_vert']  # (n, 4)
# 2D keypoints
data = np.load(index + '_pose.npz')
kp = data['kp']  # (n, 17, 3)
# 2D Segmentation masks
data = np.load(index + '_mask.npz')
mask = data['mask']  # (n, 480, 640)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --------------------------------------------
# 1. Prepare Radar Input (NO RGB USED)
# --------------------------------------------
# Combine horizontal + vertical heatmaps as 2-channel input
radar_input = np.stack([hori, vert], axis=0)  # (2, 256, 128)
radar_tensor = torch.tensor(radar_input, dtype=torch.float32).unsqueeze(0).to(device)

# Normalize
radar_tensor = radar_tensor / torch.max(radar_tensor)

# --------------------------------------------
# 2. Soft-Argmax Layer (Differentiable)
# --------------------------------------------
class SoftArgmax2D(nn.Module):
    def forward(self, heatmaps):
        B, C, H, W = heatmaps.shape
        heatmaps = heatmaps.view(B, C, -1)
        heatmaps = F.softmax(heatmaps, dim=-1)

        indices = torch.arange(H*W).float().to(heatmaps.device)
        x = (indices % W)
        y = (indices // W)

        exp_x = torch.sum(heatmaps * x, dim=-1)
        exp_y = torch.sum(heatmaps * y, dim=-1)

        coords = torch.stack([exp_x, exp_y], dim=-1)
        return coords  # (B, 17, 2)

# --------------------------------------------
# 3. Custom CNN (Encoder-Decoder + Residual)
# --------------------------------------------
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1)

    def forward(self, x):
        residual = x
        x = F.relu(self.conv1(x))
        x = self.conv2(x)
        return F.relu(x + residual)

class PoseCNN(nn.Module):
    def __init__(self):
        super().__init__()

        # Encoder
        self.enc1 = nn.Conv2d(2, 32, 3, padding=1)
        self.enc2 = nn.Conv2d(32, 64, 3, padding=1)

        # Residual block
        self.res = ResidualBlock(64)

        # Decoder
        self.dec1 = nn.Conv2d(64, 32, 3, padding=1)
        self.dec2 = nn.Conv2d(32, 32, 3, padding=1)

        # Output: 17 heatmaps
        self.out = nn.Conv2d(32, 17, 1)

        self.soft_argmax = SoftArgmax2D()

    def forward(self, x):
        x = F.relu(self.enc1(x))
        x = F.relu(self.enc2(x))
        x = self.res(x)

        x = F.relu(self.dec1(x))
        x = F.relu(self.dec2(x))

        heatmaps = self.out(x)
        coords = self.soft_argmax(heatmaps)

        return heatmaps, coords

model_cnn = PoseCNN().to(device)

# --------------------------------------------
# 4. Baseline Model (ResNet-18)
# --------------------------------------------
from torchvision.models import resnet18

class ResNetPose(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = resnet18(pretrained=False)
        self.backbone.conv1 = nn.Conv2d(2, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.backbone.fc = nn.Linear(512, 17*2)

    def forward(self, x):
        x = self.backbone(x)
        return x.view(-1, 17, 2)

model_resnet = ResNetPose().to(device)

# --------------------------------------------
# 5. Loss Function (Log Loss / BCE)
# --------------------------------------------
criterion = nn.BCEWithLogitsLoss()

# Create GT heatmaps from keypoints
def create_heatmaps(gt_kp, H=256, W=128):
    heatmaps = np.zeros((17, H, W))
    for i in range(17):
        x, y, v = gt_kp[i]
        if v > 0:
            x, y = int(x), int(y)
            if x < W and y < H:
                heatmaps[i, y, x] = 1
    return torch.tensor(heatmaps, dtype=torch.float32)

gt_kp = kp[0]
gt_heatmaps = create_heatmaps(gt_kp).unsqueeze(0).to(device)

# --------------------------------------------
# 6. Forward Pass
# --------------------------------------------
heatmaps_pred, coords_pred = model_cnn(radar_tensor)
coords_resnet = model_resnet(radar_tensor)

# Loss
loss = criterion(heatmaps_pred, gt_heatmaps)
print("Loss:", loss.item())

coords_pred = coords_pred[0].detach().cpu().numpy()
coords_resnet = coords_resnet[0].detach().cpu().numpy()

# --------------------------------------------
# 7. Metrics
# --------------------------------------------
def mae(gt, pred):
    return np.mean(np.linalg.norm(gt[:,:2] - pred, axis=1))

def pck(gt, pred, thresh=0.05):
    img_size = 256
    correct = np.linalg.norm(gt[:,:2] - pred, axis=1) < thresh * img_size
    return np.mean(correct)

def precision_recall_f1(gt, pred, thresh=5):
    dist = np.linalg.norm(gt[:,:2] - pred, axis=1)
    tp = np.sum(dist < thresh)
    fp = np.sum(dist >= thresh)
    fn = fp
    precision = tp / (tp + fp + 1e-6)
    recall = tp / (tp + fn + 1e-6)
    f1 = 2 * (precision * recall) / (precision + recall + 1e-6)
    return precision, recall, f1

def oks(gt, pred):
    oks = 0
    for i in range(17):
        d = np.linalg.norm(gt[i,:2] - pred[i])
        oks += np.exp(-d**2 / 100)
    return oks / 17

# Compute metrics
print("\n=== Custom CNN ===")
print("MAE:", mae(gt_kp, coords_pred))
print("PCK:", pck(gt_kp, coords_pred))
print("OKS:", oks(gt_kp, coords_pred))
p, r, f1 = precision_recall_f1(gt_kp, coords_pred)
print("Precision:", p, "Recall:", r, "F1:", f1)

print("\n=== ResNet-18 Baseline ===")
print("MAE:", mae(gt_kp, coords_resnet))
print("PCK:", pck(gt_kp, coords_resnet))
print("OKS:", oks(gt_kp, coords_resnet))
p, r, f1 = precision_recall_f1(gt_kp, coords_resnet)
print("Precision:", p, "Recall:", r, "F1:", f1)

# --------------------------------------------
# 8. Robustness Simulation (Noise / Occlusion)
# --------------------------------------------
noisy_input = radar_tensor + torch.randn_like(radar_tensor) * 0.1
heatmaps_noisy, coords_noisy = model_cnn(noisy_input)

coords_noisy = coords_noisy[0].detach().cpu().numpy()

print("\n=== Robustness Test (Noise) ===")
print("PCK:", pck(gt_kp, coords_noisy))

C:\Users\Joshua Miller\AppData\Roaming\Python\Python313\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\Joshua Miller\AppData\Roaming\Python\Python313\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Loss: nan

=== Custom CNN ===
MAE: nan
PCK: 0.0
OKS: nan
Precision: 0.0 Recall: 0.0 F1: 0.0

=== ResNet-18 Baseline ===
MAE: nan
PCK: 0.0
OKS: nan
Precision: 0.0 Recall: 0.0 F1: 0.0

=== Robustness Test (Noise) ===
PCK: 0.0
